# Vision Transformer from Scratch on CIFAR-10

Companion notebook for the [Vision Transformers](https://codemia.ai/courses/deep_learning_foundations/vision_transformers) lesson on Codemia.

**What you will build**: a small ViT (4 transformer layers, 64-dim embeddings) implemented from scratch in PyTorch, trained on CIFAR-10, then compared against a similarly-sized ResNet on both the full dataset and a 10x smaller subset to make the data-efficiency story concrete.

**Time**: ~60 minutes on a T4 GPU

**GPU required**: yes (CIFAR-10 + transformer = each epoch is slow on CPU). Free-tier Colab T4 is enough.

## Setup

In [ ]:
%pip install -q torch torchvision matplotlib

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')
if device == 'cuda':
    print('  GPU:', torch.cuda.get_device_name(0))

## Concept recap

A ViT treats an image as a *sequence of patch tokens*. We split the image into non-overlapping `patch_size x patch_size` squares, flatten each one into a vector, and linearly project it to an embedding of dimension `embed_dim`. From that point on, the architecture is a standard transformer encoder. We add a learnable `[CLS]` token at position 0 (whose final hidden state is the image representation), add learnable 1D positional embeddings, and stack `depth` transformer blocks.

CNNs come pre-loaded with strong inductive biases: locality (each filter sees a small window), translation equivariance (the same filter slides across the image), and hierarchy (downsampling builds receptive field). A ViT has none of these and must learn them from data. When the dataset is large enough, this freedom pays off — but on a small dataset like CIFAR-10, the CNN's built-in priors win, and the gap widens further when the data is even scarcer. We will see both effects in this notebook.

## Implementation

### Step 1 — Patch embedding

A `Conv2d` with `kernel_size = stride = patch_size` is mathematically identical to *split into non-overlapping patches and linearly project each one*. It is the most efficient way to express patch tokenization in PyTorch.

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=64):
        super().__init__()
        # Non-overlapping conv = patchify + linear-project in one op.
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.n_patches = (img_size // patch_size) ** 2
    def forward(self, x):
        # x: (B, C, H, W) -> (B, D, H/P, W/P) -> (B, D, N) -> (B, N, D)
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

# Sanity check: 32x32 image with patch_size=4 should produce 8*8 = 64 patches.
pe = PatchEmbed()
tokens = pe(torch.randn(2, 3, 32, 32))
print('patch tokens:', tokens.shape, '(expect (2, 64, 64))')

### Step 2 — Transformer block (PreNorm)

PreNorm normalizes *before* each sublayer; the residual carries the un-normed input forward. This trains more stably at depth than the original PostNorm formulation and is what every modern transformer uses.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        # batch_first=True so we feed (B, N, D) directly.
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden = int(embed_dim * mlp_ratio)
        # Standard transformer MLP: expand to 4x, GELU, project back.
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, embed_dim),
        )
    def forward(self, x):
        # PreNorm: residual connects the *un-normed* input around each sublayer.
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

### Step 3 — The full ViT

Patch embed → prepend a learnable `[CLS]` token → add learnable 1D positional embeddings → `depth` transformer blocks → LayerNorm → take the `[CLS]` token's final hidden state → linear classifier.

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 num_classes=10, embed_dim=64, depth=4, num_heads=4, mlp_ratio=4.0):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        # CLS token: a single learnable vector that participates in attention with all patches.
        # Its final hidden state aggregates information from the whole image.
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        # +1 for the CLS slot. Learnable 1D positional embeddings work fine at this scale.
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        # ViT init recipe: small truncated normal everywhere, biases at zero.
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_linear)
    def _init_linear(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                       # (B, N, D)
        cls = self.cls_token.expand(B, -1, -1)        # (B, 1, D)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        # Classify from the CLS token's final state — the standard ViT recipe.
        return self.head(self.norm(x[:, 0]))

### Step 4 — A similarly-sized ResNet baseline

We build a tiny CIFAR-style ResNet (3 stages, 2 BasicBlocks each, channels 20→40→64) and tune the channel widths so its parameter count lands within ~10% of the ViT. This way any difference we see is about *architecture*, not capacity.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        # 1x1 conv shortcut whenever shape changes (channels or stride).
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class TinyResNet(nn.Module):
    def __init__(self, num_classes=10, c=(20, 40, 64)):
        super().__init__()
        # CIFAR stem: 3x3 conv, no aggressive downsampling at the input.
        self.stem = nn.Sequential(
            nn.Conv2d(3, c[0], 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(c[0]),
            nn.ReLU(inplace=True),
        )
        self.layer1 = nn.Sequential(BasicBlock(c[0], c[0]),         BasicBlock(c[0], c[0]))
        self.layer2 = nn.Sequential(BasicBlock(c[0], c[1], 2),      BasicBlock(c[1], c[1]))
        self.layer3 = nn.Sequential(BasicBlock(c[1], c[2], 2),      BasicBlock(c[2], c[2]))
        self.head = nn.Linear(c[2], num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.head(x)

### Step 5 — Confirm the parameter counts match

If one model has 2x the parameters of the other, the comparison is meaningless. We sanity-check before training.

In [ ]:
def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

vit_ref = ViT()
rn_ref = TinyResNet()
p_vit, p_rn = n_params(vit_ref), n_params(rn_ref)
print(f'ViT       params: {p_vit:>8,}')
print(f'TinyResNet params: {p_rn:>8,}')
print(f'ratio (ViT/RN):    {p_vit / p_rn:.2f}x  (within 10% means architecture, not capacity, drives the gap)')

## Experiment 1 — Full CIFAR-10

Standard CIFAR-10 augmentation: random crop with 4-pixel padding, horizontal flip, normalize with the dataset mean/std. We train both models for the same number of epochs with the same optimizer (AdamW + cosine schedule) so the only thing that varies is the architecture.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_tf)
testset       = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_tf)
print(f'train: {len(trainset_full):,}  test: {len(testset):,}')

In [ ]:
def train_model(model, trainset, testset, epochs=12, batch_size=128, lr=3e-4, wd=0.05, label=''):
    model = model.to(device)
    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(testset,  batch_size=256,        shuffle=False, num_workers=2, pin_memory=True)
    # AdamW + cosine works well across both architectures — keeps the comparison fair.
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.train(); tot, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            logits = model(x); loss = F.cross_entropy(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0); n += x.size(0)
        sched.step()
        # Eval
        model.eval(); vtot, vn, correct = 0.0, 0, 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                logits = model(x)
                vtot += F.cross_entropy(logits, y, reduction='sum').item(); vn += x.size(0)
                correct += (logits.argmax(-1) == y).sum().item()
        history['train_loss'].append(tot / n)
        history['val_loss'].append(vtot / vn)
        history['val_acc'].append(correct / vn)
        print(f'  [{label}] epoch {epoch+1:2d}/{epochs}  train_loss={tot/n:.4f}  val_loss={vtot/vn:.4f}  val_acc={correct/vn:.4f}')
    return history

In [ ]:
EPOCHS_FULL = 12  # tuned to fit free-tier T4 — enough to see the architectures separate.

torch.manual_seed(0)
vit_full = ViT()
t0 = time.time()
h_vit_full = train_model(vit_full, trainset_full, testset, epochs=EPOCHS_FULL, label='ViT/full')
print(f'ViT (full) total: {(time.time()-t0)/60:.1f} min')

torch.manual_seed(0)
rn_full = TinyResNet()
t0 = time.time()
h_rn_full = train_model(rn_full, trainset_full, testset, epochs=EPOCHS_FULL, label='RN/full')
print(f'ResNet (full) total: {(time.time()-t0)/60:.1f} min')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(h_vit_full['train_loss'], '--', label='ViT train')
ax[0].plot(h_vit_full['val_loss'],   label='ViT val')
ax[0].plot(h_rn_full['train_loss'],  '--', label='RN train')
ax[0].plot(h_rn_full['val_loss'],    label='RN val')
ax[0].set_title('Loss — full CIFAR-10 (50k)'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(h_vit_full['val_acc'], 'o-', label='ViT')
ax[1].plot(h_rn_full['val_acc'],  's-', label='RN')
ax[1].set_title('Val accuracy — full CIFAR-10'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'final val acc — ViT: {h_vit_full["val_acc"][-1]:.4f}   RN: {h_rn_full["val_acc"][-1]:.4f}')
print(f'gap (ViT - RN): {h_vit_full["val_acc"][-1] - h_rn_full["val_acc"][-1]:+.4f}')

## Experiment 2 — 10x smaller dataset (5,000 images)

Now we take a fixed random subset of 5,000 training images (1/10 the data) and re-train both models with the same recipe. The CNN's locality bias is essentially free training data on small datasets; the ViT has to learn that bias from scratch. Watch the gap *invert*.

In [ ]:
# Fixed seed makes the subset reproducible across runs.
g = torch.Generator().manual_seed(42)
subset_idx = torch.randperm(len(trainset_full), generator=g)[:5000].tolist()
trainset_small = Subset(trainset_full, subset_idx)
print(f'small subset size: {len(trainset_small)}')

In [ ]:
EPOCHS_SMALL = 12  # same epoch count — each epoch is now 10x faster because data is 10x smaller.

torch.manual_seed(0)
vit_small = ViT()
t0 = time.time()
h_vit_small = train_model(vit_small, trainset_small, testset, epochs=EPOCHS_SMALL, label='ViT/5k')
print(f'ViT (5k) total: {(time.time()-t0)/60:.1f} min')

torch.manual_seed(0)
rn_small = TinyResNet()
t0 = time.time()
h_rn_small = train_model(rn_small, trainset_small, testset, epochs=EPOCHS_SMALL, label='RN/5k')
print(f'ResNet (5k) total: {(time.time()-t0)/60:.1f} min')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(h_vit_small['train_loss'], '--', label='ViT train')
ax[0].plot(h_vit_small['val_loss'],   label='ViT val')
ax[0].plot(h_rn_small['train_loss'],  '--', label='RN train')
ax[0].plot(h_rn_small['val_loss'],    label='RN val')
ax[0].set_title('Loss — 5k subset'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(h_vit_small['val_acc'], 'o-', label='ViT')
ax[1].plot(h_rn_small['val_acc'],  's-', label='RN')
ax[1].set_title('Val accuracy — 5k subset'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

gap_full  = h_vit_full['val_acc'][-1]  - h_rn_full['val_acc'][-1]
gap_small = h_vit_small['val_acc'][-1] - h_rn_small['val_acc'][-1]
print('---- summary ----')
print(f'full 50k:   ViT={h_vit_full["val_acc"][-1]:.4f}   RN={h_rn_full["val_acc"][-1]:.4f}    gap (ViT-RN)={gap_full:+.4f}')
print(f'small 5k:   ViT={h_vit_small["val_acc"][-1]:.4f}   RN={h_rn_small["val_acc"][-1]:.4f}    gap (ViT-RN)={gap_small:+.4f}')
print(f'gap delta (small - full): {gap_small - gap_full:+.4f}  (more negative = ViT data-efficiency cost)')

If you trained both runs cleanly, the ResNet beat the ViT on the 5k subset by a wider margin than on the full 50k set, and possibly even beat it on full CIFAR-10 too — because CIFAR-10 is *small* by ViT standards. The original ViT paper showed the crossover happens around 10M+ images. CIFAR-10 sits well below that threshold, which is exactly why this notebook is a good demo of the data-efficiency story.

## What to try next

- **Pretrain on a bigger dataset and finetune to CIFAR-10.** Even modest pretraining (e.g., on TinyImageNet or CIFAR-100) can flip the result — ViT learns the locality bias from the bigger source dataset and ports it over.
- **Longer schedules + stronger augmentation.** ViTs benefit disproportionately from MixUp, CutMix, and 100+ epochs of training. Try replacing the plain cosine schedule with warmup-then-cosine and add `RandomErasing`.
- **Smaller patches.** Drop `patch_size` from 4 to 2 (256 tokens) and watch both throughput and accuracy. Compute scales as O(N²) in token count.
- **Register tokens.** Add a few extra learnable tokens beyond `[CLS]` (the "registers" trick from Darcet et al.) and inspect whether attention maps become cleaner.
- **Hybrid stems.** Replace the patch-conv stem with a few real convolutional layers. This recovers some of the locality bias while keeping the transformer body, often a free win at small data scale.